In [3]:
from pathlib import Path
from torch.utils.data import DataLoader
from rf_learning_dataset import RFLearningDataset

train_set = RFLearningDataset(
    root_dir=DATA_ROOT / "train",
    sample_group="/sample_000001",
    normalize=EXP["normalize"],
    include_categories=EXP["include_categories"],
)

val_set = RFLearningDataset(
    root_dir=DATA_ROOT / "val",
    sample_group="/sample_000001",
    normalize=EXP["normalize"],
    include_categories=EXP["include_categories"],
)

test_set = RFLearningDataset(
    root_dir=DATA_ROOT / "test",
    sample_group="/sample_000001",
    normalize=EXP["normalize"],
    include_categories=EXP["include_categories"],
)

train_loader = DataLoader(
    train_set,
    batch_size=EXP["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

val_loader = DataLoader(
    val_set,
    batch_size=EXP["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

test_loader = DataLoader(
    test_set,
    batch_size=EXP["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print("Train:", len(train_set))
print("Val  :", len(val_set))
print("Test :", len(test_set))

batch = next(iter(train_loader))
print("input   :", batch["input"].shape)
print("label   :", batch["label"].shape)
print("baseline:", batch["baseline"].shape)
print("category:", batch["category"])
EXP = {
    "experiment_name": "tiny_baseline_cond_nonpoint_full",
    "project_root": "/home/liujia/RF_Image",

    "include_categories": ["carotid", "muscle", "phantom"],

    "batch_size": 4,
    "num_epochs": 100,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "normalize": True,
    "abs_weight": 0.1,

    "model_name": "tiny_baseline_conditioned",
    "hidden": 64,
    "seed": 20260522,
}

PROJECT_ROOT = Path(EXP["project_root"])
DATA_ROOT = PROJECT_ROOT / "Data"

CKPT_DIR = PROJECT_ROOT / "checkpoint" / EXP["experiment_name"]
METRIC_DIR = PROJECT_ROOT / "test_metrics" / EXP["experiment_name"]
VIS_DIR = PROJECT_ROOT / "vis_best_model" / EXP["experiment_name"]

CKPT_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT :", DATA_ROOT)
print("CKPT_DIR  :", CKPT_DIR)
print("METRIC_DIR:", METRIC_DIR)
print("VIS_DIR   :", VIS_DIR)



RFLearningDataset
  root_dir   : /home/liujia/RF_Image/Data/train
  samples    : 1050
  normalize  : True
  include    : {'muscle', 'phantom', 'carotid'}
  exclude    : set()
  carotid   : 350
  muscle    : 350
  phantom   : 350
RFLearningDataset
  root_dir   : /home/liujia/RF_Image/Data/val
  samples    : 225
  normalize  : True
  include    : {'muscle', 'phantom', 'carotid'}
  exclude    : set()
  carotid   : 75
  muscle    : 75
  phantom   : 75
RFLearningDataset
  root_dir   : /home/liujia/RF_Image/Data/test
  samples    : 225
  normalize  : True
  include    : {'muscle', 'phantom', 'carotid'}
  exclude    : set()
  carotid   : 75
  muscle    : 75
  phantom   : 75
Train: 1050
Val  : 225
Test : 225
input   : torch.Size([4, 1536, 16, 8, 8])
label   : torch.Size([4, 2, 16, 8, 8])
baseline: torch.Size([4, 2, 16, 8, 8])
category: ['phantom', 'muscle', 'muscle', 'phantom']
DATA_ROOT : /home/liujia/RF_Image/Data
CKPT_DIR  : /home/liujia/RF_Image/checkpoint/tiny_baseline_cond_nonpoint_full


In [4]:
from rf_models import build_model
from rf_train_utils import (
    seed_everything,
    get_device,
    count_trainable_parameters,
    train_model_jupyter,
    plot_training_curve,
)

seed_everything(EXP["seed"])

device = get_device()
print("Device:", device)

model = build_model(
    EXP["model_name"],
    in_channels=1536,
    hidden=EXP["hidden"],
    out_channels=2,
).to(device)

print(f"Trainable parameters: {count_trainable_parameters(model) / 1e6:.3f} M")

history = train_model_jupyter(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    ckpt_dir=CKPT_DIR,
    experiment_name=EXP["experiment_name"],
    num_epochs=EXP["num_epochs"],
    lr=EXP["lr"],
    weight_decay=EXP["weight_decay"],
    abs_weight=EXP["abs_weight"],
    print_every=5,
    seed=EXP["seed"],
    config=EXP,
)

Device: cuda
Trainable parameters: 0.265 M

Initial validation:
val_L1=5.573877e-01 | baseline_L1=5.573877e-01 | improvement=0.00%
Epoch 0001 | train_L1=3.114108e-01 | train_base=5.519043e-01 | train_impr= 43.58% | val_L1=2.963423e-01 | val_base=5.573877e-01 | val_impr= 46.83% | lr=1.00e-03
Epoch 0005 | train_L1=2.798186e-01 | train_base=5.518552e-01 | train_impr= 49.29% | val_L1=2.901747e-01 | val_base=5.573877e-01 | val_impr= 47.94% | lr=9.94e-04
Epoch 0010 | train_L1=2.715307e-01 | train_base=5.519964e-01 | train_impr= 50.81% | val_L1=2.949635e-01 | val_base=5.573877e-01 | val_impr= 47.08% | lr=9.76e-04
Epoch 0015 | train_L1=2.375300e-01 | train_base=5.522631e-01 | train_impr= 56.99% | val_L1=3.024913e-01 | val_base=5.573877e-01 | val_impr= 45.73% | lr=9.46e-04
Epoch 0020 | train_L1=2.085918e-01 | train_base=5.521426e-01 | train_impr= 62.22% | val_L1=3.092765e-01 | val_base=5.573877e-01 | val_impr= 44.51% | lr=9.05e-04
Epoch 0025 | train_L1=1.911574e-01 | train_base=5.519737e-01 | t

In [5]:
import torch
import pandas as pd

from rf_models import build_model
from rf_eval_utils import (
    evaluate_full_test_set,
    summarize_test_metrics,
    save_test_summaries,
    find_worse_samples,
)

from rf_visualization import (
    visualize_model_samples,
    select_indices_from_metrics_df,
)

# ============================================================
# Load best model
# ============================================================

best_model = build_model(
    EXP["model_name"],
    in_channels=1536,
    hidden=EXP["hidden"],
    out_channels=2,
).to(device)

ckpt_path = CKPT_DIR / "best_model.pth"
ckpt = torch.load(ckpt_path, map_location=device)

best_model.load_state_dict(ckpt["model"])
best_model.eval()

print("Loaded best model")
print("  epoch       :", ckpt["epoch"])
print("  best val L1 :", ckpt["best_val_l1"])
print("  ckpt path   :", ckpt_path)

# ============================================================
# Full test evaluation
# ============================================================

df_test = evaluate_full_test_set(
    model=best_model,
    dataset=test_set,
    device=device,
    batch_size=EXP["batch_size"],
    save_csv_path=METRIC_DIR / "test_per_sample_metrics.csv",
)

overall, cat_summary = summarize_test_metrics(df_test)

save_test_summaries(
    df=df_test,
    metric_dir=METRIC_DIR,
    overall=overall,
    cat_summary=cat_summary,
    prefix="test",
)

# ============================================================
# Worse samples
# ============================================================

worse_complex = find_worse_samples(
    df_test,
    metric="complex",
    top_k=20,
)

worse_abs = find_worse_samples(
    df_test,
    metric="abs",
    top_k=20,
)

# 也保存一下，方便后面查
worse_complex.to_csv(METRIC_DIR / "test_worse_complex_top20.csv", index=False)
worse_abs.to_csv(METRIC_DIR / "test_worse_abs_top20.csv", index=False)

# ============================================================
# Visualization: random/category samples
# ============================================================

vis_random = visualize_model_samples(
    model=best_model,
    dataset=test_set,
    device=device,
    save_dir=VIS_DIR / "random_by_category",
    indices=None,
    n_per_category=3,
    categories=EXP["include_categories"],
    view="xz",
    slice_index=None,
    db_min=-60,
    show=False,
    prefix="random",
)

# ============================================================
# Visualization: worst complex samples
# ============================================================

worst_indices = select_indices_from_metrics_df(
    dataset=test_set,
    df=df_test,
    top_k=9,
    metric="complex_improvement",
    ascending=True,
)

vis_worst = visualize_model_samples(
    model=best_model,
    dataset=test_set,
    device=device,
    save_dir=VIS_DIR / "worst_complex",
    indices=worst_indices,
    view="xz",
    slice_index=None,
    db_min=-60,
    show=False,
    prefix="worst_complex",
)

print("\nEvaluation package finished.")
print("Metric dir:", METRIC_DIR)
print("Vis dir   :", VIS_DIR)
print("Random visualizations:", len(vis_random))
print("Worst visualizations :", len(vis_worst))

Loaded best model
  epoch       : 4
  best val L1 : 0.28841361037471835
  ckpt path   : /home/liujia/RF_Image/checkpoint/tiny_baseline_cond_nonpoint_full/best_model.pth
Saved per-sample metrics to: /home/liujia/RF_Image/test_metrics/tiny_baseline_cond_nonpoint_full/test_per_sample_metrics.csv

================ Overall test summary ================
Samples: 225

[Complex L1]
pred mean       : 3.8619e+03
baseline mean   : 6.3340e+03
mean improvement: 45.43%
better rate     : 95.11%

[Envelope abs L1]
pred mean       : 3.2582e+03
baseline mean   : 6.5403e+03
mean improvement: 49.65%
better rate     : 99.11%

================ Per-category summary ================


,n,complex_pred_mean,complex_base_mean,complex_improvement_mean,complex_better_rate,abs_pred_mean,abs_base_mean,abs_improvement_mean,abs_better_rate
category,,,,,,,,,
carotid,75,6421.756019,8853.747764,0.375610,0.880000,5171.982212,8805.461160,0.430026,0.973333
muscle,75,3833.075984,7649.877194,0.491515,0.986667,3418.182589,8249.515830,0.554033,1.000000
phantom,75,1330.718676,2498.391657,0.495924,0.986667,1184.479923,2565.977287,0.505401,1.000000


Saved test summaries to: /home/liujia/RF_Image/test_metrics/tiny_baseline_cond_nonpoint_full
complex worse samples: 11


,path,category,pred_complex_l1,base_complex_l1,complex_improvement,complex_better,pred_abs_l1,base_abs_l1,abs_improvement,abs_better
31,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,33367.843750,16654.449219,-1.003539,False,28251.541016,15645.277344,-0.805755,False
56,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,43911.507812,22148.236328,-0.982619,False,19615.154297,19800.744141,0.009373,True
36,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,52268.957031,29001.410156,-0.802290,False,24731.070312,24740.470703,0.000380,True
191,/home/liujia/RF_Image/Data/test/phantom/RF0005...,phantom,13611.999023,8542.150391,-0.593510,False,4290.880371,11631.220703,0.631089,True
2,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,12540.438477,8185.803223,-0.531974,False,7307.200684,8064.032715,0.093853,True
50,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,26048.003906,21248.251953,-0.225889,False,18837.769531,18237.437500,-0.032918,False
12,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,7782.527344,6349.586426,-0.225675,False,5361.729492,5375.907715,0.002637,True
21,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,22778.345703,20525.160156,-0.109777,False,14685.186523,15921.531250,0.077652,True
22,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,1794.732422,1688.536865,-0.062892,False,1227.270386,1476.075073,0.168558,True
93,/home/liujia/RF_Image/Data/test/muscle/RF00038...,muscle,9156.255859,8627.460938,-0.061292,False,5601.830566,6820.783203,0.178712,True


abs worse samples: 2


,path,category,pred_complex_l1,base_complex_l1,complex_improvement,complex_better,pred_abs_l1,base_abs_l1,abs_improvement,abs_better
31,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,33367.843750,16654.449219,-1.003539,False,28251.541016,15645.277344,-0.805755,False
50,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,26048.003906,21248.251953,-0.225889,False,18837.769531,18237.437500,-0.032918,False


Selected indices: [0, 1, 2, 75, 76, 77, 150, 151, 152]
Saved: /home/liujia/RF_Image/vis_best_model/tiny_baseline_cond_nonpoint_full/random_by_category/001_random_carotid_RF000486_carotid_test_patch001.png
  complex L1 pred/base: 8.4619e+03 / 1.3601e+04
  abs     L1 pred/base: 8.3177e+03 / 1.2952e+04
Saved: /home/liujia/RF_Image/vis_best_model/tiny_baseline_cond_nonpoint_full/random_by_category/002_random_carotid_RF000486_carotid_test_patch002.png
  complex L1 pred/base: 9.0081e+03 / 1.9757e+04
  abs     L1 pred/base: 6.5078e+03 / 2.3112e+04
Saved: /home/liujia/RF_Image/vis_best_model/tiny_baseline_cond_nonpoint_full/random_by_category/003_random_carotid_RF000486_carotid_test_patch003.png
  complex L1 pred/base: 1.2540e+04 / 8.1858e+03
  abs     L1 pred/base: 7.3072e+03 / 8.0640e+03
Saved: /home/liujia/RF_Image/vis_best_model/tiny_baseline_cond_nonpoint_full/random_by_category/004_random_muscle_RF000386_muscle_test_patch001.png
  complex L1 pred/base: 3.1203e+03 / 5.9643e+03
  abs     L